In [1]:
import pandas as pd
import numpy as np
from functools import reduce
import seaborn as sns
import yaml

In [2]:
with open("../config.yaml", "r") as file:
    config = yaml.safe_load(file)

In [3]:
df_cc = pd.read_csv(config['data']['clean']['file1'], quotechar='"', sep = ";")
df_hu = pd.read_csv(config['data']['clean']['file2'], quotechar='"', sep = ";")
df_pp = pd.read_csv(config['data']['clean']['file3'], quotechar='"', sep = ";")
df_qq = pd.read_csv(config['data']['clean']['file4'], quotechar='"', sep = ";")
df_rr = pd.read_csv(config['data']['clean']['file5'], quotechar='"', sep = ";")
df_sd = pd.read_csv(config['data']['clean']['file6'], quotechar='"', sep = ";")
df_ss = pd.read_csv(config['data']['clean']['file7'], quotechar='"', sep = ";")
df_tg = pd.read_csv(config['data']['clean']['file8'], quotechar='"', sep = ";")
df_tn = pd.read_csv(config['data']['clean']['file9'], quotechar='"', sep = ";")
df_tx = pd.read_csv(config['data']['clean']['file10'], quotechar='"', sep = ";")

In [4]:
all_data = reduce(lambda  left,right: pd.merge(left,right,on=['date'],how='outer'), [df_cc,df_hu,df_pp, df_qq,df_rr,df_sd,df_ss,df_tg,df_tn,df_tx])
all_data.date  = pd.to_datetime(all_data.date, format='%Y-%m-%d')

In [ ]:
all_data["year"] = all_data["date"].dt.year
all_data["day_of_year"] = all_data["date"].dt.dayofyear
all_data["doy_sin"] = np.sin(2 * np.pi * all_data["day_of_year"] / 365.25)
all_data["doy_cos"] = np.cos(2 * np.pi * all_data["day_of_year"] / 365.25)
all_data["tx_lag1"] = all_data["tx"].shift(1)
all_data["tx_lag3"] = all_data["tx"].shift(3)
all_data["tx_lag7"] = all_data["tx"].shift(7)

all_data["tn_lag1"] = all_data["tn"].shift(1)
all_data["tg_lag1"] = all_data["tg"].shift(1)

all_data["pp_lag1"] = all_data["pp"].shift(1)
all_data["hu_lag1"] = all_data["hu"].shift(1)
all_data["rr_lag1"] = all_data["rr"].shift(1)
all_data["cc_lag1"] = all_data["cc"].shift(1)
all_data = all_data.dropna()

In [ ]:
all_data

In [ ]:
df_hw = all_data.drop(columns='date')

In [ ]:
# sns.heatmap(df_ad.corr(), annot=True)
df_hw.corr()

In [ ]:
df_hw = df_hw.drop(columns=['pp','rr','sd','tg','tn'])

In [ ]:
test = df_hw[df_hw.year > 2021]
target = df_hw[df_hw.year <= 2021]
y_train = target.tx
y_test  = test.tx
X_train = target.drop(columns=['tx','year'])
X_test  = test.drop(columns=['tx','year'])

In [ ]:
# target= df_ad.tx
# features = df_ad.drop(columns=['tg','tx','tn'])

In [ ]:
target

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
# X_train, X_test, y_train, y_test = train_test_split(features, target, test_size = 0.20,  random_state=0)

In [ ]:
X_train.shape
normalizer = MinMaxScaler()
normalizer.fit(X_train)
X_train_norm = normalizer.transform(X_train)
X_test_norm = normalizer.transform(X_test)
X_train_norm = pd.DataFrame(X_train_norm, columns = X_train.columns)
X_test_norm = pd.DataFrame(X_test_norm, columns = X_test.columns)

In [ ]:
knn = KNeighborsRegressor()
knn.fit(X_train_norm, y_train)
print(f"The accuracy of the model is {knn.score(X_test_norm, y_test): .2f}")

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor,AdaBoostRegressor, GradientBoostingRegressor

forest = RandomForestRegressor(n_estimators=100,
                             max_depth=20)
forest.fit(X_train_norm, y_train)
y_pred_test_rf = forest.predict(X_test_norm)


# print(f"MAE {mean_absolute_error(y_pred_test, y_test): .2f}") # mean(abs(error)) = mean(abs(y_test - y_pred_test))
# print(f"RMSE, {np.sqrt(mean_squared_error(y_pred_test, y_test)): .2f}") # sqrt( mean( (y_test - y_pred_test)^2 ) ) # b0, b1, b2...
print(f"R2 score, {forest.score(X_test_norm, y_test): .2f}")

In [ ]:
df_snow = all_data.drop(columns='date')
df_snow["tn_lag3"] = df_snow["tn"].shift(3)
df_snow["tn_lag7"] = df_snow["tn"].shift(7)
df_snow.dropna()